# HỆ THỐNG QUẢN LÝ BÁN LẺ

## Khởi tạo và Thêm dữ liệu

- Tạo database QL_BAN_HANG.

- Tạo 2 collection: sanpham và donhang.

- Dùng lệnh insert_many để thêm dữ liệu giả lập.

In [1]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.3 MB/s eta 0:00:00


In [2]:
from datetime import datetime
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from google.colab import userdata

uri = userdata.get('MONGODB_URI')

client = MongoClient(uri, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Kết nối MongoDB thành công!")
except Exception as e:
    print(e)

Kết nối MongoDB thành công!


In [5]:
db = client['QL_BAN_HANG']

collection_sp = db['sanpham']
collection_dh = db['donhang']

print("đã tạo thành công db và collection!\n")

đã tạo thành công db và collection!



In [6]:
sp = [
    {
        "masp": "sp01", "tensp": "iPhone 15 Pro", "giagoc": 30000, "giaban": 28000,
        "danhmuc": ["Dien thoai", "Apple", "Giam gia"],
        "danhgia": [{"user": "Khach A", "sao": 5}, {"user": "Khach B", "sao": 3}]
    },
    {
        "masp": "sp02", "tensp": "MacBook Air M2", "giagoc": 25000, "giaban": 25000,
        "danhmuc": ["Laptop", "Apple"],
        "danhgia": [{"user": "Khach C", "sao": 4}]
    },
    {
        "masp": "sp03", "tensp": "Samsung S24 Ultra", "giagoc": 26000, "giaban": 24000,
        "danhmuc": ["Dien thoai", "Samsung", "Giam gia"],
        "danhgia": [{"user": "Khach A", "sao": 5}]
    }
]

dh = [
    {
        "madon": "dh01", "khachhang": "Khach A", "trangthai": "Hoan thanh", "tongtien": 52000,
        "chitiet": [{"masp": "sp01", "soluong": 1}, {"masp": "sp03", "soluong": 1}]
    },
    {
        "madon": "dh02", "khachhang": "Khach B", "trangthai": "Hoan thanh", "tongtien": 56000,
        "chitiet": [{"masp": "sp01", "soluong": 2}]
    },
    {
        "madon": "dh03", "khachhang": "Khach A", "trangthai": "Dang giao", "tongtien": 25000,
        "chitiet": [{"masp": "sp02", "soluong": 1}]
    },
    {
        "madon": "dh04", "khachhang": "Khach C", "trangthai": "Da huy", "tongtien": 25000,
        "chitiet": [{"masp": "sp02", "soluong": 1}]
    }
]

collection_sp.insert_many(sp)
collection_dh.insert_many(dh)
print("insert thành công!\n")

insert thành công!



## Truy vấn cơ bản & Toán tử Mảng

- Tìm các sản phẩm thuộc CẢ 2 danh mục: "Dien thoai" VÀ "Giam gia" (Chỉ hiển thị tên sản phẩm và danh mục, ẩn _id).

- Tìm các sản phẩm có ít nhất 1 đánh giá 5 sao từ khách hàng.

- Tìm những sản phẩm đang được sale (có giaban < giagoc) bằng toán tử biểu thức so sánh chéo.

In [7]:
dm_dienThoai_giamGia = collection_sp.find(
    {
        "danhmuc": {"$all": ["Dien thoai", "Giam gia"]}
    },
    {
        "tensp": 1, "danhmuc": 1, "_id": 0,
    }
)

for i in dm_dienThoai_giamGia:
    print(i)


{'tensp': 'iPhone 15 Pro', 'danhmuc': ['Dien thoai', 'Apple', 'Giam gia']}
{'tensp': 'Samsung S24 Ultra', 'danhmuc': ['Dien thoai', 'Samsung', 'Giam gia']}


In [10]:
dg_5sao = collection_sp.find(
    {
        "danhgia": {
            "$elemMatch":{
                "sao": 5
            }
        }
    },
    {"tensp": 1, "danhgia": 1, "_id": 0}
)

for i in dg_5sao:
    print(i)

{'tensp': 'iPhone 15 Pro', 'danhgia': [{'user': 'Khach A', 'sao': 5}, {'user': 'Khach B', 'sao': 3}]}
{'tensp': 'Samsung S24 Ultra', 'danhgia': [{'user': 'Khach A', 'sao': 5}]}


In [11]:
dang_sale = collection_sp.find(
    {
        "$expr": {
            "$lt": ["$giaban", "$giagoc"]
        }
    },
    {"tensp": 1, "giagoc": 1, "giaban": 1, "_id": 0}
)

for i in dang_sale:
    print(i)

{'tensp': 'iPhone 15 Pro', 'giagoc': 30000, 'giaban': 28000}
{'tensp': 'Samsung S24 Ultra', 'giagoc': 26000, 'giaban': 24000}


##  Cập nhật & Xóa dữ liệu

- Thêm tag "Moi ve" vào mảng danhmuc của sản phẩm có mã sp02 (Đảm bảo chạy nhiều lần không bị trùng lặp tag).

- Dùng kỹ thuật *array_filters* để sửa đánh giá của user "Khach B" trong sản phẩm sp01 thành 4 sao.

- Xóa các đơn hàng có trạng thái là "Da huy".

## Aggregation Pipeline

- Dùng `$group` và `$sum` để tính Tổng doanh thu theo từng trangthai đơn hàng.

- (`$out`, `$lookup`): Tìm xem Khách hàng nào mua nhiều số lượng sản phẩm nhất (Tính tổng số lượng của tất cả các sản phẩm nằm trong mảng chitiet của mọi đơn hàng).